# Project: BBLF AI Selector v2 
# Section: Data Preparation
## Sub Section: Pre Tournament

Goal: Join all previous seasons' individual ball by ball datasets into each player's fantasy points in each game


# 0. Prerequisties

In [46]:
# 0. Prerequistes
import pandas as pd
import numpy as np
import os
from concurrent.futures import ProcessPoolExecutor

os.getcwd()
directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/'

# 1. Data Extraction & Raw Data Constuction

In [47]:
# Pull in all_matches csv file
raw_df = pd.read_csv(os.path.join(directory,'data/add_data_created/past_season_data/all_matches.csv'), low_memory=False)
print(raw_df.shape)

# Create loop to extract additional matches for BBL 24/25 season and join to raw df
with ProcessPoolExecutor() as executor:
    for i in range(1443057, 1443100):
        game_num = (i - 1443057 + 1)
        print(game_num)
        try:
            # Create relative path for file extraction
            file_path = f'data/add_data_created/past_season_data/{i}.csv'

            # Pull in individual game data
            add_csv = pd.read_csv(os.path.join(directory, file_path), low_memory=False)

            # Attach individual game data to raw df
            raw_df = pd.concat([raw_df, add_csv])

            # Delete individual dataset
            del add_csv

        except Exception as e:
            # Handle the error (optional)
            print(f"Error processing {i}: {e}")
            # Skip to the next iteration
            continue

print(raw_df.shape)

# Converting season column to integer format
raw_df_clean = raw_df.copy()
raw_df_clean["season"] = raw_df_clean['season'].str.split("/", n=1, expand = True)[0].astype(int) - 2010

(133547, 22)
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
Error processing 1443077: [Errno 2] No such file or directory: 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/past_season_data/1443077.csv'
22
23
24
25
26
27
Error processing 1443083: [Errno 2] No such file or directory: 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/past_season_data/1443083.csv'
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
(143112, 22)


# 2. Fantasy Points of each player per game

In [48]:
# a. Create function to aggregate bowl by bowl data to overall innings scorecard summary

# List of all the unique BBL match ids    
match_id_list = raw_df_clean.match_id.unique()

# Create empty dataframe to hold fantasy point outputs from the for loop
fant_pts_table = pd.DataFrame()
fant_pts_table["match_id"] = []
fant_pts_table["player"] = []
fant_pts_table["fantasy_point_bat"] = []
fant_pts_table["fantasy_point_bowl"] = []
fant_pts_table["fantasy_point"] = []

# For loop to collect the player fantasy points for each game and add to fantasy points table
with ProcessPoolExecutor() as executor:
    for x in match_id_list:

        # select specific match id from all ids
        game_df = raw_df[raw_df["match_id"] == x]

        #game_df.to_csv('data/python_datasets/bbl_23_24_final_data.csv')

        game_df_clean = game_df[["match_id", "innings", "ball", "striker", "non_striker","bowler", "runs_off_bat","wides","noballs","wicket_type","player_dismissed","batting_team", "bowling_team"]]

        # 2a. Batting (1. Total Runs, 2. Strike Rate, 3. Run Bonus (50 & 100))
        # 1. Total Runs
        game_bat_df = game_df_clean[["match_id", "striker", "runs_off_bat", "batting_team", "bowling_team"]]
        game_bat_df = game_bat_df.rename(columns = {"batting_team": "team", "bowling_team": "opposition"})
        game_bat_df_agg = game_bat_df.groupby(['match_id', 'striker', "team", "opposition"], as_index=False).agg(
        total_runs=('runs_off_bat',"sum"),
        total_balls=("runs_off_bat","count"))

        # 2. Strike Rate
        game_bat_df_agg["strike_rate_elig_f"] = np.where((game_bat_df_agg["total_runs"] >= 20), 1, 0)
        game_bat_df_agg["strike_rate"] = game_bat_df_agg["total_runs"]/game_bat_df_agg["total_balls"]*100

        sr_conditions = [
            (game_bat_df_agg['strike_rate'] < 120),
            (game_bat_df_agg['strike_rate'] >= 120) & (game_bat_df_agg['strike_rate'] < 130),
            (game_bat_df_agg['strike_rate'] >= 130) & (game_bat_df_agg['strike_rate'] < 140),
            (game_bat_df_agg['strike_rate'] >= 140) & (game_bat_df_agg['strike_rate'] < 150),
            (game_bat_df_agg['strike_rate'] >= 150) & (game_bat_df_agg['strike_rate'] < 160),
            (game_bat_df_agg['strike_rate'] >= 160)
        ]

        sr_group = [0,1,2,3,4,5]

        game_bat_df_agg["strike_rate_group"] = np.select(sr_conditions, sr_group)

        # 3. Run Bonus
        rb_conditions = [
            (game_bat_df_agg['total_runs'] < 50),
            (game_bat_df_agg['total_runs'] >= 50) & (game_bat_df_agg['total_runs'] < 100),
            (game_bat_df_agg['total_runs'] >= 100)
        ]

        rb_group = [0,1,2]

        game_bat_df_agg["run_bonus_group"] = np.select(rb_conditions, rb_group)

        # 4. Batting Fantasy Points
        game_bat_df_agg["fantasy_point_bat"] = game_bat_df_agg["total_runs"] + (game_bat_df_agg["strike_rate_elig_f"]*game_bat_df_agg["strike_rate_group"]*5) + game_bat_df_agg["run_bonus_group"]*10 

        # 2b. Bowling (1. Wickets, 2. Economy, 3. Wicket Bonus (3), 4. Maiden , 5. Dot Ball 6. Extras)
        # 1. Wickets, Dot Balls, Extras
        game_bowl_df = game_df[["match_id", "innings", "ball", "striker", "non_striker","bowler", "runs_off_bat","wides","noballs","wicket_type","player_dismissed","batting_team", "bowling_team"]]
        game_bowl_df = game_bowl_df.rename(columns = {"batting_team": "opposition", "bowling_team": "team"})
        game_bowl_df["ball"] = game_bowl_df["ball"].astype(str) 

        game_bowl_df_over = game_bowl_df["ball"].str.split(".", n=1, expand = True)
        game_bowl_df["over"] = game_bowl_df_over[0]
        game_bowl_df["ball"] = game_bowl_df_over[1]

        game_bowl_df["bowl_wicket"] = np.where((game_bowl_df["wicket_type"] == "runout" ) |(game_bowl_df["wicket_type"].isnull()) , 0 , 1)
        game_bowl_df["dot_ball_f"] = np.where(game_bowl_df["runs_off_bat"] == 0, 1, 0)
        game_bowl_df["wides"] = game_bowl_df["wides"].fillna(0)
        game_bowl_df["noballs"] = game_bowl_df["noballs"].fillna(0)
        game_bowl_df["bowl_extra"] = (game_bowl_df["wides"] + game_bowl_df["noballs"])
        game_bowl_df["elig_bowl"] = np.where((game_bowl_df["bowl_extra"] == 0) ,1, 0)

        game_bowl_df_agg = game_bowl_df.groupby(["match_id", "bowler", "team", "opposition"], as_index=False).agg(
        wickets = ("bowl_wicket", "sum"),
        runs = ("runs_off_bat", "sum"),
        ball_cnt = ("elig_bowl", "sum"),
        dot_ball_cnt = ("dot_ball_f", "sum"),
        extra_cnt = ("bowl_extra", "sum")
        )

        # 2. Economy
        game_bowl_df_agg["econ_elig_f"] = np.where((game_bowl_df_agg["ball_cnt"] >= 18), 1, 0)
        game_bowl_df_agg["econ"] = (game_bowl_df_agg["runs"]/game_bowl_df_agg["ball_cnt"]*6)

        econ_conditions = [
            (game_bowl_df_agg['econ'] <= 4),
            (game_bowl_df_agg['econ'] > 4) & (game_bowl_df_agg['econ'] <= 5),
            (game_bowl_df_agg['econ'] > 5) & (game_bowl_df_agg['econ'] <= 6),
            (game_bowl_df_agg['econ'] > 6) & (game_bowl_df_agg['econ'] <= 7),
            (game_bowl_df_agg['econ'] > 7) & (game_bowl_df_agg['econ'] <= 8),
            (game_bowl_df_agg['econ'] > 8)
        ]

        econ_group = [5,4,3,2,1,0]
        game_bowl_df_agg["econ_bonus_group"] = np.select(econ_conditions, econ_group)

        # 3. Wicket Bonus
        wb_conditions = [
            (game_bowl_df_agg['wickets'] < 3),
            (game_bowl_df_agg['wickets'] >= 3) & (game_bowl_df_agg['wickets'] < 6),
            (game_bowl_df_agg['wickets'] >= 6) & (game_bowl_df_agg['wickets'] < 9),
            (game_bowl_df_agg['wickets'] >= 9)
        ]

        wb_group = [0,1,2,3]
        game_bowl_df_agg["wicket_bonus_group"] = np.select(wb_conditions, wb_group)

        # 4. Maiden
        game_bowl_over_df_agg = game_bowl_df.groupby(["match_id", "bowler", "over"]).agg(
        dot_ball_cnt = ("dot_ball_f", "sum")
        )

        game_bowl_over_df_agg["maiden_f"] = np.where((game_bowl_over_df_agg["dot_ball_cnt"] == 6), 1, 0)

        game_bowl_maiden_agg = game_bowl_over_df_agg.groupby(["match_id", "bowler"]).agg(
        maiden_cnt = ("maiden_f", "sum")    
        )

        game_bowl_df_agg = pd.merge(game_bowl_df_agg, game_bowl_maiden_agg, left_on = "bowler", right_on = "bowler", how = "left")

        # 5. Bowling Fantasy Points 
        game_bowl_df_agg["fantasy_point_bowl"] = game_bowl_df_agg["wickets"]*20 + game_bowl_df_agg["dot_ball_cnt"] + game_bowl_df_agg["extra_cnt"]*(-1) + (game_bowl_df_agg["econ_bonus_group"]*game_bowl_df_agg["econ_elig_f"]*5) + game_bowl_df_agg["wicket_bonus_group"]*10 + game_bowl_df_agg["maiden_cnt"]*15 

        # 2c. Overall game player fantasy points
        game_bat_play_sum = game_bat_df_agg[["match_id", "striker", "team", "opposition", "fantasy_point_bat", "total_runs", "strike_rate_elig_f", "strike_rate_group", "run_bonus_group"]].rename(columns={"striker": "player"})
        game_bowl_play_sum = game_bowl_df_agg[["match_id", "bowler", "team", "opposition", "fantasy_point_bowl", "wickets", "dot_ball_cnt", "extra_cnt", "econ_bonus_group", "econ_elig_f", "wicket_bonus_group", "maiden_cnt"]].rename(columns={"bowler": "player"})
        game_play_fantasy_pts_df = pd.merge(game_bat_play_sum, game_bowl_play_sum, left_on = ["player", "match_id"], right_on = ["player", "match_id"], how = "outer")
        game_play_fantasy_pts_df["team"] = np.where(game_play_fantasy_pts_df["team_x"].isnull(), game_play_fantasy_pts_df["team_y"], game_play_fantasy_pts_df["team_x"])
        game_play_fantasy_pts_df["opp"] = np.where(game_play_fantasy_pts_df["opposition_x"].isnull(), game_play_fantasy_pts_df["opposition_y"], game_play_fantasy_pts_df["opposition_x"])
        game_play_fantasy_pts_df.drop(columns = ["team_x", "team_y", "opposition_x", "opposition_y"], inplace = True)
        game_play_fantasy_pts_df.fillna(0)
        game_play_fantasy_pts_df["fantasy_point"] = game_play_fantasy_pts_df["fantasy_point_bat"] + game_play_fantasy_pts_df["fantasy_point_bowl"]
        fant_pts_table = pd.concat([fant_pts_table, game_play_fantasy_pts_df])

# Final clean up of fantasy points table
fant_pts_table['match_id'] = fant_pts_table['match_id'].astype(int)

# 2b. Identify missing players (who played the game but got 0 points)

In [49]:
# Aggregate fantasy points table by team per game and get count of players per team
match_team_agg = fant_pts_table.groupby(["match_id", "team"], as_index=False).agg(
    player_count = ("player", "count")
)

# Identify match id - teams with less than 11 players and store in a list
match_team_few_players = match_team_agg[match_team_agg["player_count"] < 11]
match_team_few_players["match_id"] = match_team_few_players["match_id"].astype(int)
match_team_few_players["match_id_team"] = match_team_few_players["match_id"].astype(str) + "_" + match_team_few_players["team"]
match_team_few_players_list = match_team_few_players["match_id_team"].tolist()
print(len(match_team_few_players_list))

# Create function to extract match info from csv and add player to fantasy points table
with ProcessPoolExecutor() as executor:
    for i in match_team_few_players_list:
        try:
            # Split match id and team
            match_id = i.split("_")[0]
            team = i.split("_")[1]
            print(f'Processing match_id: {match_id}, team: {team}')
        
            # Pull in individual game data
            # Create relative path for file extraction
            file_path = f'data/add_data_created/past_season_data/bbl_v2_data/{match_id}_info.csv'

            # Pull in individual game info
            expected_cols = ['blank', 'type', 'team', 'name', 'blank2']
            match_csv = pd.read_csv(
                os.path.join(directory, file_path),
                header=None,
                names=expected_cols,
                skiprows=1,           # drop the bad header row
                engine='python'
            )

            # Retrieve all player names for the game
            full_play = match_csv[(match_csv["type"] == "player") & (match_csv["team"] == team)]['name'].tolist()

            # Extract existing players from fantasy points table for the match and team
            exist_play = fant_pts_table[(fant_pts_table["match_id"] == int(match_id)) & (fant_pts_table["team"] == team)]["player"].tolist()
            opposition = fant_pts_table[(fant_pts_table["match_id"] == int(match_id)) & (fant_pts_table["team"] == team)]["opp"].iloc[0]

            # Identify missing players
            missing_play = list(set(full_play) - set(exist_play))

            # Add missing players to fantasy points table with 0 points
            for p in missing_play:
                new_row = {
                    "match_id": int(match_id),
                    "player": p,
                    "fantasy_point_bat": 0,
                    "fantasy_point_bowl": 0,
                    "fantasy_point": 0,
                    "total_runs": 0,
                    "strike_rate_elig_f": 0,
                    "strike_rate_group": 0,
                    "run_bonus_group": 0,
                    "wickets": 0, 
                    "dot_ball_cnt": 0,
                    "extra_cnt": 0, 
                    "econ_bonus_group": 0, 
                    "econ_elig_f": 0, 
                    "wicket_bonus_group": 0, 
                    "maiden_cnt": 0,
                    "team": team,
                    "opp": opposition,
                }

                fant_pts_table = pd.concat([fant_pts_table, pd.DataFrame([new_row])], ignore_index=True)

        except Exception as e:
            # Handle the error (optional)
            print(f"Error processing {i}: {e}")
            # Skip to the next iteration
            continue

# Check:
match_team_agg_final = fant_pts_table.groupby(["match_id", "team"], as_index=False).agg(
    player_count = ("player", "count")
)
print(match_team_agg_final[match_team_agg_final["player_count"] < 11])

C:\Users\dilan\AppData\Local\Temp\ipykernel_19776\2960258043.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  match_team_few_players["match_id"] = match_team_few_players["match_id"].astype(int)
C:\Users\dilan\AppData\Local\Temp\ipykernel_19776\2960258043.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  match_team_few_players["match_id_team"] = match_team_few_players["match_id"].astype(str) + "_" + match_team_few_players["team"]


320
Processing match_id: 524915, team: Sydney Sixers
Processing match_id: 524920, team: Hobart Hurricanes
Processing match_id: 524921, team: Melbourne Renegades
Processing match_id: 524921, team: Perth Scorchers
Processing match_id: 524928, team: Melbourne Renegades
Processing match_id: 524930, team: Melbourne Stars
Processing match_id: 524933, team: Sydney Sixers
Processing match_id: 524933, team: Sydney Thunder
Processing match_id: 524935, team: Melbourne Stars
Processing match_id: 524938, team: Brisbane Heat
Processing match_id: 524939, team: Brisbane Heat
Processing match_id: 524940, team: Hobart Hurricanes
Processing match_id: 524940, team: Melbourne Renegades
Processing match_id: 524942, team: Melbourne Stars
Processing match_id: 524943, team: Perth Scorchers
Processing match_id: 524945, team: Sydney Sixers
Processing match_id: 571232, team: Melbourne Renegades
Processing match_id: 571233, team: Sydney Sixers
Processing match_id: 571234, team: Brisbane Heat
Processing match_id: 5

# 3. Add Key Index Columns to main table

In [50]:
# Create match ids primary index table
match_primary_index = raw_df_clean[["match_id", "start_date", "season", "venue", "innings", "batting_team", "bowling_team"]]
match_primary_index = match_primary_index[match_primary_index.innings == 1]

match_primary_index_agg = match_primary_index.groupby(["match_id", "start_date", "season", "venue", "innings", "batting_team", "bowling_team"], as_index=False).agg(
    count = ("season", "count"),
    )

match_primary_index_agg =  match_primary_index_agg[["match_id", "start_date", "season", "venue", "batting_team", "bowling_team"]].rename(columns={"batting_team": "first_bat_team", "bowling_team": "first_bowl_team"})

# Join primary index and fantasy points table
fant_pts_df = pd.merge(fant_pts_table, match_primary_index_agg, left_on = ["match_id"], right_on = ["match_id"], how = "left")

# 4. Save Data

In [51]:
fant_pts_df.to_csv(os.path.join(directory,'data/python_data/fant_point_play_tbl.csv'))
